In [1]:
import numpy as np
rng = np.random.default_rng(2026)
prices = 100 * np.exp(np.cumsum(rng.normal(0.0005, 0.02, 250)))
rets = np.diff(prices) / prices[:-1]           # 简单收益率
log_rets = np.diff(np.log(prices))             # 对数收益率

def annualize(ret, n=250):                     # 年化均值
    return ret.mean() * n
def ann_vol(ret, n=250):                       # 年化波动
    return ret.std(ddof=1) * np.sqrt(n)
def downside_dev(ret, target=0.0):             # 下行半方差
    neg = ret[ret < target]
    return np.sqrt(np.mean(neg**2))
def max_drawdown(p):                           # 最大回撤
    cummax = np.maximum.accumulate(p)
    return np.min(p / cummax - 1)

print("年化收益", round(annualize(rets), 4))
print("年化波动", round(ann_vol(rets), 4))
print("下行半方差", round(downside_dev(rets), 4))
print("最大回撤", round(max_drawdown(prices), 4))

年化收益 0.57
年化波动 0.3392
下行半方差 0.02
最大回撤 -0.1955


In [2]:
rf, E_rp, sig_p, A = 0.03, 0.10, 0.18, 4.0
y_star = (E_rp - rf) / (A * sig_p**2)        # 最优风险资产比例
E_rc = rf + y_star * (E_rp - rf)             # 组合期望收益
sig_c = y_star * sig_p                        # 组合波动
sharpe = (E_rp - rf) / sig_p                  # 夏普比率

print("y* =", round(y_star, 3))
print("E(rc) =", round(E_rc, 4))
print("sigma_c =", round(sig_c, 4))
print("Sharpe =", round(sharpe, 3))

y* = 0.54
E(rc) = 0.0678
sigma_c = 0.0972
Sharpe = 0.389


In [ ]:
2.1 Day1 收益率与风险度量
2.1.1 知识点模块
收益率的两套口径
- 简单收益率：R_t = (P_t − P_{t−1}) / P_{t−1}，直观，但多期不能直接相加，组合加权才准确。
- 对数收益率：r_t = ln(P_t / P_{t−1})，多期可相加（r 总 = Σr_t），时间上更对称，建模常用。
- 换算：日→年化 = 均值×250（或几何复利 e^{μ×250}−1）；波动年化 = 日标准差×√250。
风险的三个度量
- 标准差（波动率）：对上下波动一视同仁，是最常用的风险度量。
- 下行半方差（Downside Deviation）：只统计低于目标（如 0）的损失——投资者更在意亏，而非涨。
- 最大回撤（Max Drawdown）：净值从历史高点跌到低点的最大幅度——衡量"最坏的时候亏多少"。





2.1.3 验收项答案
验收项
答案要点
简单收益率和对数收益率在什么场景各用哪个？
描述单期涨跌、算组合收益用简单收益率；做多期加总、时间序列建模用对数收益率（可加性）。
年化波动为什么乘 √250 而不是 250？
波动与 √时间 成正比（随机游走假设）：日方差×250 → 日标准差×√250。
下行半方差和标准差的核心区别？
标准差惩罚涨和跌；下行半方差只惩罚亏损方向，符合"风险=亏损"的直觉。
最大回撤代表什么、有什么用？
代表从高点最坏跌多少；回撤 50% 需要涨 100% 才回本，用于评估策略最坏情况和风险承受匹配。

In [ ]:
2.2 Day2 风险厌恶与资本配置线
2.2.1 知识点模块
效用函数：收益与风险的取舍
- U = E(r) − ½·A·σ²：把"想要收益"和"讨厌波动"放一个式子里。A 是风险厌恶系数：A 越大越保守，A=0 无所谓，A<0 爱冒险。
- 投资者选择使 U 最大的组合——这就是"最优"的定义。
资本配置线（CAL）
- 把资金按比例 y 分给风险组合 P、1−y 分给无风险资产 rf：E(r_c) = rf + y·(E(r_p) − rf)，σ_c = y·σ_p。
- 消去 y 得直线：E(r_c) = rf + Sharpe·σ_c——斜率就是夏普比率，线上每个点代表一个 y。
- 最优比例：y* = (E(r_p) − rf) / (A·σ_p²)。分子是"风险溢价"，分母是"风险×厌恶"——溢价越高买越多，波动越大或越厌恶买越少。



y* 公式的分子分母分别是什么含义？
分子=风险溢价 E(rp)−rf（补偿，越多越买）；分母=A·σ_p²（波动×厌恶，越痛越少买）。
y*>1 说明什么？
说明最优选择是"借钱加杠杆"买风险组合（超过 100% 资金配置），发生在风险溢价很高或厌恶很低时。
CAL 的斜率为什么是夏普比率？
线上每单位波动获得的超额收益 = (E(rp)−rf)/σ_p = Sharpe，衡量"每冒一分险换多少回报"。

In [ ]:
2.3 Day3 最优风险组合与有效前沿
2.3.1 知识点模块
组合的方差
- 两资产：σ_p² = w₁²σ₁² + w₂²σ₂² + 2w₁w₂·Cov(r₁,r₂)。协方差（相关性）是分散化的核心：相关系数越低，组合风险越低。
有效前沿与两个特殊点
- 可行域：所有可能权重组合的（σ, E(r)）集合；有效前沿：同波动下收益最高（或同收益下波动最低）的边界。
- GMV（全局最小方差组合）：前沿上波动最小的点。
- 切点组合 T：从 rf 引直线与前沿相切，斜率（夏普）最大的组合——加上无风险资产后，所有投资者都应持有 T + rf 的组合（两基金分离定理）。


预期输出：两行——GMV 权重（波动最小的一组权重）与切点权重（夏普最大的权重）。种子无关（确定性优化），数值以你的输出为准；GMV 的波动应明显低于单个资产，切点的夏普应高于任一单资产。
2.3.3 验收项答案
验收项
答案要点
为什么相关系数越低组合风险越低？
两资产同涨同跌（ρ 高）时分散无效；ρ 低时一方的下跌常被另一方的上涨抵消，组合波动被"平均"掉。
GMV 和切点组合的区别？
GMV 只追求波动最小、不关心收益；切点组合在引入无风险资产后夏普最大，是理性投资者唯一需要持有的风险组合。
两基金分离定理说了什么？
所有投资者的最优组合都是"切点组合 T + 无风险资产"的线性组合，区别只在 y*（怎么分），风险组合本身一样。


In [3]:
#Day3 GMV 与最大夏普组合（scipy）
import numpy as np
from scipy.optimize import minimize

mu = np.array([0.08, 0.12, 0.15])
cov = np.array([[0.10**2, 0.005, 0.002],
                [0.005, 0.18**2, 0.008],
                [0.002, 0.008, 0.22**2]])

def stats(w):
    w=np.asarray(w)
    return w@mu,w@cov@w

def min_var(w):
    return stats(w)[1]

cons=({'type':'eq','fun':lambda w:w.sum()-1})
bnds=[(0,1)]*3
w_gmv=minimize(min_var,[1/3]*3,bounds=bnds,constraints=cons).x

def neg_sharpe(w):
    er,var=stats(w)
    return -(er-0.03)/np.sqrt(var)

w_tan = minimize(neg_sharpe, [1/3]*3, bounds=bnds, constraints=cons).x
print("GMV权重", np.round(w_gmv, 3), "波动", round(np.sqrt(stats(w_gmv)[1]), 4))
print("切点权重", np.round(w_tan, 3), "夏普", round(-neg_sharpe(w_tan), 3))

GMV权重 [0.764 0.112 0.124] 波动 0.0919
切点权重 [0.5   0.226 0.273] 夏普 0.765


In [ ]:
3.1 Day4 协方差、β 与系统性风险
3.1.1 知识点模块
- 协方差与相关性：度量两个资产收益一起变动的方向与强度；相关性 = 协方差 ÷ (σ₁σ₂)，取值 [−1, 1]。
- β 定义：β = Cov(rᵢ, rₘ) / Var(rₘ)——资产对市场整体波动的敏感度。β=1 跟市场同步，β>1 放大市场波动（进攻型），β<1 抗跌（防御型）。
- 分散化只消除非系统性风险：组合中股票越多，公司特有的风险（非系统性）被抵消，但市场共有的风险（系统性）无法消除——这正是 CAPM 只给系统性风险定价的原因（Day5）。
估计 β 接近真实值 1.2（相关系数较高）；组合方差随股票数增加而下降，且下降幅度递减——说明分散化有极限，极限就是系统性风险。
3.1.3 验收项答案
验收项
答案要点
β 与相关系数的区别？
相关系数衡量"关联强度"（无量纲）；β 衡量"对市场波动的敏感倍数"（有方向的斜率），β 还受自身波动影响。
为什么股票加到 50 只后波动几乎不再降？
非系统性风险已基本被分散完，剩下的只有市场共有的系统性风险——它无法靠加股票消除。



预期输出：估计 β 接近真实值 1.2（相关系数较高）；组合方差随股票数增加而下降，且下降幅度递减——说明分散化有极限，极限就是系统性风险。
3.1.3 验收项答案
验收项
答案要点
β 与相关系数的区别？
相关系数衡量"关联强度"（无量纲）；β 衡量"对市场波动的敏感倍数"（有方向的斜率），β 还受自身波动影响。
为什么股票加到 50 只后波动几乎不再降？
非系统性风险已基本被分散完，剩下的只有市场共有的系统性风险——它无法靠加股票消除。

In [4]:
#Day4 估计 β 与分散化效应
import numpy as np
rng = np.random.default_rng(7)
rm = rng.normal(0.01, 0.02, 200)                # 市场收益
ri = 0.008 + 1.2 * rm + rng.normal(0, 0.015, 200)  # 真β=1.2

beta = np.cov(ri, rm)[0, 1] / np.var(rm)
corr = np.corrcoef(ri, rm)[0, 1]
print("估计β =", round(beta, 3))
print("与市场相关系数 =", round(corr, 3))

n = np.arange(1, 51)                            # 1~50只等权股票
port_var = [np.var(rm) * 0.3**2 / i for i in n]  # 非系统部分随n递减
print("20只股票组合方差≈", round(port_var[19], 6))
print("50只股票组合方差≈", round(port_var[49], 6))

估计β = 1.221
与市场相关系数 = 0.824
20只股票组合方差≈ 1e-06
50只股票组合方差≈ 1e-06


In [ ]:
3.2 Day5 CAPM 与证券市场线
3.2.1 知识点模块
CAPM 公式
- E(rᵢ) = rf + βᵢ·(E(rₘ) − rf)——资产的期望收益只取决于它的 β，与自身波动无关。
- 含义：只有系统性风险被定价。非系统性风险可被分散掉，市场不会为"可免费消除的风险"付钱。
证券市场线（SML）与 α
- SML：以 β 为横轴、期望收益为纵轴的直线，斜率 = 市场风险溢价 E(rₘ)−rf。市场组合 β=1，无风险资产 β=0。
- α = 实际预期收益 − SML 上的公平收益：α>0 → 被低估（买入）；α<0 → 被高估（卖出）。CAPM 下均衡状态所有 α=0。


预期输出：β=0.5→0.065；β=1.0→0.10；β=1.5→0.135；α = 0.09 − 0.086 = 0.004（>0，按 CAPM 被低估）。
3.2.3 验收项答案
验收项
答案要点
为什么波动大但 β 小的资产定价不高？
其高波动主要来自非系统性风险，分散化后消失，市场不补偿；定价只看 β（系统性敏感度）。
SML 与 CAL 的区别？
CAL 是某个组合随 y 变化的配置线（横轴 σ）；SML 描述所有资产的均衡定价（横轴 β），是全市场的基准线。
α>0 在 CAPM 框架下说明什么？
资产收益高于其 β 应得的公平收益，被低估；反之 α<0 被高估。均衡时无 α（市场已定价充分）。

In [7]:
rf, E_rm = 0.03, 0.10

def sml(beta):
    return rf + beta * (E_rm - rf)

print("β=0.5 期望收益", round(sml(0.5), 4))
print("β=1.0 期望收益", round(sml(1.0), 4))
print("β=1.5 期望收益", round(sml(1.5), 4))

alpha = 0.09 - sml(0.8)     # 某基金实际预期收益9%，β=0.8
print("α =", round(alpha, 4))

β=0.5 期望收益 0.065
β=1.0 期望收益 0.1
β=1.5 期望收益 0.135
α = 0.004


In [ ]:
3.3 Day6 β 估计与 CAPM 检验
3.3.1 知识点模块
- β 的实证估计：把资产收益对市场收益回归 rᵢ − rf = α + β·(rₘ − rf) + ε。斜率就是 β，截距是 α 的估计。
- 检验思路：CAPM 的预测——截距 α 应为 0（无超额），β 应能完全解释截面收益差异；实证中 α≠0 的异象（Day9）构成对 CAPM 的挑战。
- CAPM 的局限：β 不稳定（随时间变）、市场组合不可观测（用指数近似）、现实存在规模/价值等 β 解释不了的溢价（Day8 多因子）。

预期输出：α≈0.002（接近设定的 0.002）、β≈0.9、残差波动≈0.012（接近设定的特有波动）。回归把收益拆成"市场驱动部分 + 公司特有残差"。
3.3.3 验收项答案
验收项
答案要点
回归的截距和斜率分别对应 CAPM 的什么？
斜率=β（系统性敏感度）；截距=α（超额收益，CAPM 预测为 0）。
残差波动代表什么？
收益中市场因子解释不了的部分——非系统性风险（公司特有），CAPM 认为它不该被定价。
CAPM 实证检验的主要困难？
市场组合不可观测（只能用指数近似）；β 随时间变化不稳定；样本与数据频率影响估计。
预期输出：α≈0.002（接近设定的 0.002）、β≈0.9、残差波动≈0.012（接近设定的特有波动）。回归把收益拆成"市场驱动部分 + 公司特有残差"。
3.3.3 验收项答案
验收项
答案要点
回归的截距和斜率分别对应 CAPM 的什么？
斜率=β（系统性敏感度）；截距=α（超额收益，CAPM 预测为 0）。
残差波动代表什么？
收益中市场因子解释不了的部分——非系统性风险（公司特有），CAPM 认为它不该被定价。
CAPM 实证检验的主要困难？
市场组合不可观测（只能用指数近似）；β 随时间变化不稳定；样本与数据频率影响估计。

In [8]:
#Day6 OLS 估计 α 与 β
import numpy as np
rng = np.random.default_rng(11)
rm = rng.normal(0.01, 0.02, 250)                 # 市场超额收益
ri = 0.002 + 0.9 * rm + rng.normal(0, 0.012, 250)

X = np.column_stack([np.ones_like(rm), rm])      # [1, rm]
beta_hat = np.linalg.lstsq(X, ri, rcond=None)[0]
print("alpha(截距) =", round(beta_hat[0], 4))
print("beta(斜率) =", round(beta_hat[1], 3))

resid = ri - X @ beta_hat
print("残差波动 =", round(resid.std(ddof=2), 4))

alpha(截距) = 0.0022
beta(斜率) = 0.942
残差波动 = 0.0122


In [ ]:
3.4 Day7 APT 套利定价理论
3.4.1 知识点模块
- 因子模型：rᵢ = E(rᵢ) + βᵢ₁F₁ + βᵢ₂F₂ + … + εᵢ——收益由几个宏观因子（增长、通胀、利率等）驱动，β 是因子暴露。
- APT 结论：E(rᵢ) = λ₀ + βᵢ₁λ₁ + βᵢ₂λ₂ + …——期望收益由因子暴露×因子风险溢价决定，不需要市场组合、不需要效用假设。
- 与 CAPM 的关系：CAPM 是 APT 的特例（市场是唯一因子）；APT 更一般，但没说"因子是哪些"——要靠研究者自己选（Day8）。


预期输出：β≈0.8、残差≈0.02；APT 期望收益 = 0.02 + 0.8×0.03 ≈ 0.044。
3.4.3 验收项答案
验收项
答案要点
APT 与 CAPM 的关键差异？
APT 允许任意多个因子且不依赖市场组合与投资者效用；CAPM 是单因子（市场）特例。APT 没指定因子，需要研究者确定。
什么是套利组合？
零成本、零风险、却有确定正收益的组合——APT 假设市场不存在这种机会，由此推导出线性定价关系。

In [9]:
#Day7 单因子模型回归与 APT 定价
import numpy as np
rng = np.random.default_rng(13)
F = rng.normal(0, 1, 300)                        # 宏观因子(标准化)
r = 0.02 + 0.8 * F + rng.normal(0, 0.02, 300)    # 因子暴露=0.8

X = np.column_stack([np.ones_like(F), F])
coef = np.linalg.lstsq(X, r, rcond=None)[0]
print("因子暴露(β) =", round(coef[1], 3))
print("残差波动 =", round((r - X @ coef).std(ddof=2), 4))

lam0, lamF = 0.02, 0.03                          # 无风险溢价 + 因子溢价
E_r = lam0 + coef[1] * lamF
print("APT期望收益 =", round(E_r, 4))

因子暴露(β) = 0.799
残差波动 = 0.0205
APT期望收益 = 0.044


In [ ]:
3.5 Day8 多因子模型（Fama-French）
3.5.1 知识点模块
- 为什么要多因子：单因子（市场）解释不了两个异象——小盘股长期跑赢大盘股、低市净率（价值）股跑赢成长股。
- 三因子模型：E(rᵢ) = rf + β·(市场) + s·SMB + h·HML。SMB=小减大（规模因子），HML=高账面市值比减低（价值因子）；s、h 是因子暴露。
- 因子投资：把"因子"当作可交易的风格——通过多空组合（买入暴露高的、卖出暴露低的）获取因子溢价；这是 Smart Beta 与量化多头的理论基础。

预期输出：α≈0.001、市场 β≈1.0、SMB≈0.3、HML≈0.5——回归成功还原设定的因子暴露；α 接近 0 说明三因子已经把收益"解释干净"。
3.5.3 验收项答案
验收项
答案要点
SMB 与 HML 各代表什么？
SMB（Small Minus Big）= 小市值减大市值组合的收益，规模因子；HML（High Minus Low）= 高账面市值比减低账面市值比，价值因子。
加入 SMB、HML 后 α 变小说明什么？
原来的"超额收益"其实是因子暴露的补偿——多因子把收益来源归因清楚了，剩下的 α 才是真正的选股能力。
因子投资怎么落地？
构造多空因子组合（买高暴露卖低暴露）持有，赚因子溢价；注意拥挤与衰减（对应黑箱第10章）。


In [10]:
import numpy as np
rng = np.random.default_rng(17)
n = 500
MKT = rng.normal(0.008, 0.04, n)
SMB = rng.normal(0.002, 0.02, n)
HML = rng.normal(0.003, 0.02, n)
r = 0.001 + 1.0*MKT + 0.3*SMB + 0.5*HML + rng.normal(0, 0.01, n)

X = np.column_stack([np.ones(n), MKT, SMB, HML])
coef = np.linalg.lstsq(X, r, rcond=None)[0]
print("alpha =", round(coef[0], 4))
print("市场β =", round(coef[1], 2))
print("SMB =", round(coef[2], 2))
print("HML =", round(coef[3], 2))

alpha = 0.0005
市场β = 1.01
SMB = 0.31
HML = 0.48


In [ ]:
4.1 Day9 有效市场假说与行为金融
4.1.1 知识点模块
EMH 三形态
- 弱式：历史价格信息已反映在价格中，技术分析无效（收益自相关应为 0）。
- 半强式：所有公开信息已反映，基本面分析无效（事件公布瞬间价格到位）。
- 强式：所有信息（含内幕）已反映——现实中不成立，内幕交易能赚钱。
市场异象与行为偏差
- 异象：动量（过去赢家继续赢）、价值（低估值跑赢）、规模（小盘跑赢）、公告后漂移——EMH 解释不了的规律。
- 行为偏差：过度自信（交易过度）、损失厌恶（不愿止损）、羊群效应、锚定——导致价格系统性偏离。
- 实践结论：市场"大体有效、局部可被利用"——CAPM 是基准，异象与行为金融解释偏差来源。



预期输出：随机游走序列自相关≈0（接近 0，历史收益无法预测未来 → 支持弱式有效）；动量序列自相关明显为正（可预测 → 存在动量异象）。
4.1.3 验收项答案
验收项
答案要点
收益自相关接近 0 意味着什么？
今天的收益与昨天的收益无关——历史价格不能预测未来，符合弱式有效市场。
动量异象与弱式有效的矛盾？
动量说明历史价格"能"预测（自相关为正），与弱式有效冲突；行为金融用"反应不足/羊群"解释它。
半强式有效下怎么评价基本面分析？
公开信息已定价，传统基本面分析难获超额；超额只可能来自信息优势或行为偏差利用（量化角度）。

In [11]:
#Day9 弱式有效检验：收益自相关
import numpy as np
rng = np.random.default_rng(23)
rets = rng.normal(0.0004, 0.015, 500)           # 弱式有效：随机游走
lag1 = np.corrcoef(rets[:-1], rets[1:])[0, 1]
print("滞后1自相关 =", round(lag1, 4))

# 反例：构造可预测(动量)序列
predictable = 0.5 * rets[:-1] + rng.normal(0, 0.01, len(rets)-1)
print("动量序列自相关 =", round(np.corrcoef(predictable[:-1], predictable[1:])[0, 1], 3))

滞后1自相关 = 0.0858
动量序列自相关 = 0.044


In [ ]:
5. 阶段四 估值（Day10-13）
5.1 Day10 债券估值
5.1.1 知识点模块
- 定价：价格 = 所有票息与面值的现值之和。YTM（到期收益率）是使价格等于现金流的折现率——债的"内部收益率"。
- 久期（Duration）：价格对利率的敏感度，约等于"加权平均回本时间"。修正久期 D_mod：ΔP/P ≈ −D_mod·Δy——利率升，价格跌。
- 凸性（Convexity）：价格-利率关系的弯曲程度——凸性大，利率下降时涨得比久期预测多、上升时跌得比预测少（对持有人有利）。
- 收益率曲线：不同期限的到期收益率连线；正常形态向上倾斜（期限溢价），倒挂常被看作衰退信号。

预期输出：价格≈108.11（票息高于市场利率 → 溢价）；久期 8 左右（10年期债的麦考利久期小于期限）；凸性为正；利率 +1% 时价格约跌 −久期×0.01（约 −8%）。具体数值以运行为准。
5.1.3 验收项答案
验收项
答案要点
为什么票息高于市场利率的债是溢价交易？
未来现金流更值钱，按市场利率折现后的现值高于面值——价格高于 100。
久期衡量什么、怎么用它管理利率风险？
衡量价格对利率的敏感度（ΔP/P≈−D·Δy）；利率预期上升就缩短久期（卖长债买短债），预期下降就拉长久期。
凸性为什么对债券持有人有利？
价格-利率曲线凸向持有人：降息时涨得多、加息时跌得少，久期近似低估了降息收益、高估了加息损失。


In [12]:
#Day10 债券价格、久期与凸性
import numpy as np

def bond_price(c, y, T, F=100):
    t = np.arange(1, T+1)
    return np.sum(c*F/(1+y)**t) + F/(1+y)**T

def duration(c, y, T, F=100):
    t = np.arange(1, T+1)
    pv = c*F/(1+y)**t
    pvF = F/(1+y)**T
    P = pv.sum() + pvF
    return (t @ pv + T*pvF) / P

def convexity(c, y, T, F=100):
    t = np.arange(1, T+1)
    pv = c*F/(1+y)**t
    pvF = F/(1+y)**T
    P = pv.sum() + pvF
    return (t*(t+1) @ pv + T*(T+1)*pvF) / (P*(1+y)**2)

c, y, T = 0.05, 0.04, 10
print("价格 =", round(bond_price(c, y, T), 2))
print("久期 =", round(duration(c, y, T), 3))
print("凸性 =", round(convexity(c, y, T), 3))
print("利率+1%价格变动约 =", round(-duration(c, y, T)*0.01, 4))

价格 = 108.11
久期 = 8.191
凸性 = 77.482
利率+1%价格变动约 = -0.0819


In [ ]:
5.2 Day11 股票估值
5.2.1 知识点模块
- DDM（股利贴现模型）：股票价值 = 未来所有股利的现值——股票是"现金流"而非"价格"的折现。
- 戈登增长模型：P = D₁/(r − g)，要求 g<r。三个输入：下期股利、要求收益率（可来自 CAPM）、永续增长率。
- 自由现金流与倍数法：FCFE/FCFF 折现适合不派息的公司；PE = 价格/每股盈利，适合可比公司横向比价——注意盈利质量与周期性。
- 敏感性：估值对 r 和 g 极敏感（分母差一点，价格差很多）——估值给区间，不给精确值。


预期输出：g=0.05 → 50.0；g=0.03 → 35.71；g=0.07 → 83.33——增长率从 3% 到 7%，价格几乎翻倍，直观展示估值对 g 的敏感；PE 法 60.0。
5.2.3 验收项答案
验收项
答案要点
戈登模型的适用条件？
稳定派息、增长率 g 恒定且低于要求收益率 r；高增长或不分红公司不适用（要用多阶段 DDM 或 FCF 折现）。
为什么估值对 g 如此敏感？
r−g 在分母上，g 微调导致分母大幅变化；因此估值应给区间并用敏感性分析，而不是报一个精确数字。
PE 倍数法的局限？
需可比公司；盈利为负或周期性波动时失真；没考虑增长与风险差异（同 PE 不同质量）。

In [13]:
#Day11 戈登模型与敏感性
def gordon(D1, r, g):
    return D1 / (r - g)

D1, r, g = 2.5, 0.10, 0.05
print("戈登模型价格 =", round(gordon(D1, r, g), 2))

for gg in [0.03, 0.05, 0.07]:
    print("g =", gg, "价格 =", round(gordon(D1, r, gg), 2))

eps, pe = 4.0, 15.0
print("市盈率法目标价 =", round(eps*pe, 2))

戈登模型价格 = 50.0
g = 0.03 价格 = 35.71
g = 0.05 价格 = 50.0
g = 0.07 价格 = 83.33
市盈率法目标价 = 60.0


In [ ]:
5.3 Day12 期权定价
5.3.1 知识点模块
二叉树模型
- 把到期时间切成若干步，每步股价只走"上 u / 下 d"两条路；从到期价值 max(S−K,0) 用风险中性概率逐级折现回推。
- 核心思想：期权可用标的股票+无风险债券复制，因此定价与投资者风险偏好无关。
Black-Scholes 公式
- C = S·N(d₁) − K·e^{−rT}·N(d₂)；d₁、d₂ 由 S、K、T、r、σ 决定。关键输入是波动率 σ（唯一不可直接观测的输入）。
- 二叉树步数足够多时收敛到 BS——同一逻辑、两种实现。
买卖平价：C − P = S − K·e^{−rT}——看涨与看跌之间存在强制关系，可用于套利与换算。


预期输出：三期二叉树看涨≈8.1（与 BS 接近但略偏）；BS 看涨≈10.45、看跌≈7.50；平价校验两侧≈2.96 相等——二叉树与 BS 收敛、平价关系成立。
5.3.3 验收项答案
验收项
答案要点
为什么期权定价不需要知道投资者风险偏好？
期权可由股票+债券精确复制，复制成本唯一确定；风险中性概率只是定价工具，不是真实概率。
BS 公式哪个输入最关键、最难获得？
波动率 σ——唯一不可直接观测的输入；用历史波动或隐含波动率（由市场价格反推）。
买卖平价怎么用？
C−P = S−K·e^{−rT}：知道看涨可算看跌（反之亦然）；违反平价即出现套利机会。

In [15]:
#Day12 二叉树与 Black-Scholes
import numpy as np
from math import log, sqrt, exp
from scipy.stats import norm

def binom(S, K, T, r, sig, n=3, is_call=True):
    dt = T/n
    u = exp(sig*sqrt(dt)); d = 1/u
    p = (exp(r*dt) - d) / (u - d)                 # 风险中性概率
    S_T = S * u**np.arange(n, -1, -1) * d**np.arange(0, n+1)
    V = np.maximum(S_T-K, 0) if is_call else np.maximum(K-S_T, 0)
    for _ in range(n):                            # 逐级回推
        V = exp(-r*dt) * (p*V[1:] + (1-p)*V[:-1])
    return V[0]

def bs(S, K, T, r, sig, is_call=True):
    d1 = (log(S/K) + (r+0.5*sig**2)*T) / (sig*sqrt(T))
    d2 = d1 - sig*sqrt(T)
    if is_call:
        return S*norm.cdf(d1) - K*exp(-r*T)*norm.cdf(d2)
    return K*exp(-r*T)*norm.cdf(-d2) - S*norm.cdf(-d1)

S, K, T, r, sig = 100, 100, 1, 0.03, 0.20
print("三期二叉树看涨 =", round(binom(S,K,T,r,sig,3,True), 3))
print("BS看涨 =", round(bs(S,K,T,r,sig,True), 3))
print("BS看跌 =", round(bs(S,K,T,r,sig,False), 3))
print("平价校验 C-P =", round(bs(S,K,T,r,sig,True)-bs(S,K,T,r,sig,False), 3),
      "S-Ke^(-rT) =", round(S-K*exp(-r*T), 3))

三期二叉树看涨 = 8.916
BS看涨 = 9.413
BS看跌 = 6.458
平价校验 C-P = 2.955 S-Ke^(-rT) = 2.955


In [ ]:
5.4 Day13 希腊字母与期权策略
5.4.1 知识点模块
- Delta（Δ）：期权价格对股价的敏感度（0~1，看涨）；约等于"到期实值概率"。Delta 对冲 = 按 Δ 买卖股票。
- Gamma（Γ）：Delta 对股价的敏感度——衡量对冲需要多频繁调整；平值期权 Gamma 最大。
- Theta（Θ）：时间流逝的损耗（期权是"耗损资产"，到期越近跌得越快）。
- Vega（ν）：对波动率的敏感度——波动率上升期权变贵（买方受益）。
- 两个实战策略：保护性看跌（持股+买看跌，封住下行、保留上行）；备兑看涨（持股+卖看涨，收权利金但放弃部分上行）。


预期输出：平值看涨 Delta≈0.54（略大于 0.5）、Gamma 正值（平值最大）、Theta 为负（时间损耗）、Vega 为正（约 0.37/每波动率点）、Rho 小正。
5.4.3 验收项答案
验收项
答案要点
为什么平值期权的 Gamma 最大？
平值时 Delta 对股价变化最敏感（从约 0.5 快速摆动），对冲调整最频繁；深度实值/虚值 Delta 变化慢。
Theta 为负意味着什么？
时间流逝让期权贬值——期权买方在"买时间"，卖方在"卖时间"；临近到期衰减加速。
保护性看跌的损益特征？
股价大跌时看跌赔付封住损失下限，上涨时收益不封顶但减去权利金成本——用成本换尾部保护。

In [16]:
#Day13 BS 希腊字母
import numpy as np
from math import log, sqrt, exp
from scipy.stats import norm

def bs_greeks(S, K, T, r, sig):
    d1 = (log(S/K) + (r+0.5*sig**2)*T) / (sig*sqrt(T))
    d2 = d1 - sig*sqrt(T)
    delta = norm.cdf(d1)
    gamma = norm.pdf(d1) / (S*sig*sqrt(T))
    theta = -(S*norm.pdf(d1)*sig)/(2*sqrt(T)) - r*K*exp(-r*T)*norm.cdf(d2)
    vega = S*norm.pdf(d1)*sqrt(T) / 100
    rho = K*T*exp(-r*T)*norm.cdf(d2) / 100
    return delta, gamma, theta, vega, rho

d, g, t, v, rho = bs_greeks(100, 100, 1, 0.03, 0.20)
print("Delta =", round(d, 3), " Gamma =", round(g, 4))
print("Theta =", round(t, 4), " Vega =", round(v, 3), " Rho =", round(rho, 3))

Delta = 0.599  Gamma = 0.0193
Theta = -5.3804  Vega = 0.387  Rho = 0.505


In [ ]:
6. 阶段五 评价（Day14-15）
6.1 Day14 业绩评价指标
6.1.1 知识点模块
- 夏普比率：Sharpe = (E(rp) − rf) / σp——每单位总风险的超额收益；适合衡量组合整体，但不区分系统性/非系统性。
- 特雷诺比率：Treynor = (E(rp) − rf) / βp——每单位系统性风险的超额收益；适合已充分分散的组合（β 才有意义）。
- 詹森 α：α = (E(rp) − rf) − βp·(E(rm) − rf)——超额收益中超出 CAPM 定价的部分，衡量选股能力。
- 信息比率：IR = 主动收益均值 / 跟踪误差——相对基准的"每单位偏离风险的超额"，量化基金最常用。



预期输出：四个指标数值以运行为准；逻辑上组合日收益均值高于基准 → 詹森 α 为正、信息比率为正。
6.1.3 验收项答案
验收项
答案要点
夏普与特雷诺各自适用什么场景？
夏普看总风险，适合衡量组合整体表现（含未分散风险）；特雷诺只看系统性风险，适合已充分分散的组合，与基准比较更公平。
詹森 α 与信息比率的区别？
α 是绝对超额（超出 CAPM 定价的部分）；信息比率是超额除以跟踪误差，衡量超额的质量与稳定性。
只看高收益不看指标有什么风险？
高收益可能来自高波动/高 β/运气——指标把风险、基准、稳定性纳入，防止被收益数字误导。

In [ ]:
#Day14 四大绩效指标
import numpy as np
rng = np.random.default_rng(29)
rp = rng.normal(0.009, 0.02, 250)     # 组合日收益
rb = rng.normal(0.007, 0.015, 250)    # 基准日收益

ann = lambda r: r.mean()*250
vol = lambda r: r.std(ddof=1)*np.sqrt(250)
sharpe = (ann(rp) - 0.03) / vol(rp)
beta = np.cov(rp, rb)[0, 1] / np.var(rb)
treynor = (ann(rp) - 0.03) / beta
jensen = (ann(rp) - 0.03) - beta*(ann(rb) - 0.03)
te = (rp - rb).std(ddof=1)*np.sqrt(250)
ir = (rp - rb).mean()*250 / te

print("夏普 =", round(sharpe, 3))
print("特雷诺 =", round(treynor, 3))
print("詹森α =", round(jensen, 4))
print("信息比率 =", round(ir, 3))

In [ ]:
6.2 Day15 组合管理实战（全流程）
6.2.1 知识点模块
全流程五步
1. 资产配置：定无风险+风险组合的比例（Day2 的 y*）；风险组合内部按有效前沿思路配权重（Day3）。
2. 构建组合：用目标权重生成组合收益序列。
3. 再平衡：定期（如每 20 日）把实际权重拉回目标权重——纪律性交易，防止漂移。
4. 业绩归因：用 Day14 的指标拆解收益来源：市场 β 部分 vs 主动 α 部分。
5. 复盘：对照目标检查偏离、记录问题——组合管理是循环，不是一次性任务。
6.2.2 可复跑代码




预期输出：最优权重（偏向夏普更高的资产2）、组合年化与波动、信息比率为正（相对基准）、再平衡 13 次。数值以运行为准。
6.2.3 验收项答案
验收项
答案要点
为什么需要定期再平衡？
资产涨跌会让实际权重偏离目标（涨多的占比变大、风险结构改变）；再平衡用纪律拉回目标，控制风险漂移并低买高卖。
信息比率为正说明什么？
组合相对基准每承担一单位跟踪误差获得正超额——主动管理是"值"的；为负则不如直接持有基准。
业绩归因的第一步怎么做？
把组合收益拆成"β 贡献（市场部分）+ α 贡献（主动部分）"：β 部分=β×基准超额，α=詹森 α——先分清运气与能力。

In [17]:
#Day15 完整组合流程
import numpy as np
rng = np.random.default_rng(31)

# 1) 两风险资产 + 无风险
mu = np.array([0.10, 0.14]); sig = np.array([0.16, 0.24])
rho = 0.3
cov = np.array([[sig[0]**2, rho*sig[0]*sig[1]],
                [rho*sig[0]*sig[1], sig[1]**2]])

# 2) 网格搜索最大夏普权重
ws = np.linspace(0, 1, 101)
def sharpe_of(w):
    er = w*mu[0] + (1-w)*mu[1]
    var = w**2*cov[0,0] + (1-w)**2*cov[1,1] + 2*w*(1-w)*cov[0,1]
    return (er - 0.03)/np.sqrt(var)
w_best = ws[np.argmax([sharpe_of(w) for w in ws])]
print("最优风险组合权重(资产1) =", round(w_best, 3))

# 3) 生成组合收益序列并评价
a1 = rng.normal(mu[0]/250, sig[0]/np.sqrt(250), 250)
a2 = rng.normal(mu[1]/250, sig[1]/np.sqrt(250), 250)
port = w_best*a1 + (1-w_best)*a2
rm = rng.normal(0.007, 0.015, 250)                # 基准
print("组合年化 =", round(port.mean()*250, 4), " 波动 =", round(port.std()*np.sqrt(250), 4))
ir = (port-rm).mean()*250 / ((port-rm).std()*np.sqrt(250))
print("相对基准信息比率 =", round(ir, 3))

# 4) 再平衡：每20日回归目标权重（此处只记录计划次数）
print("再平衡次数 =", len(range(0, 250, 20)))

最优风险组合权重(资产1) = 0.58
组合年化 = 0.3595  波动 = 0.1342
相对基准信息比率 = -4.926
再平衡次数 = 13


In [ ]:
6.3 全期自检汇总表
Day
核心验收项
达标判断（答案出处）
D1
会算年化收益/年化波动/下行波动/回撤
代码跑通并解释 √250 口径、下行半方差只罚亏损
D2
会用 y* 公式并解释 CAL
能算出 y*≈0.540 并说出 y*>1 是加杠杆
D3
复现 GMV 与切点组合
scipy 输出两组权重，解释 GMV 最小波动、切点夏普最大
D4
估计 β 并解释分散化极限
β≈1.2 且组合方差随 n 递减趋平——只剩系统性风险
D5
按 β 定价并算 α
SML 上 β=0.8 公平收益 8.6%，实际 9% → α=0.004 被低估
D6
OLS 估计 α、β 并解释残差
斜率=β、截距=α，残差=非系统性风险
D7
单因子回归 + APT 定价
E(r)=λ0+βλF，因子暴露≈0.8、溢价 0.03 → 0.044
D8
三因子回归并解释载荷
α 归零、SMB/HML 载荷还原——超额被因子解释
D9
自相关检验 + EMH 含义
随机游走自相关≈0 → 弱式有效；动量序列为正 → 异象
D10
算价格/久期/凸性
价格≈108 溢价交易；久期≈8；凸性为正、利率+1% 约跌 8%
D11
戈登模型与敏感性
g=3%/5%/7% → 35.7/50/83.3，解释分母 r−g 的杠杆效应
D12
二叉树与 BS 收敛、平价成立
两法结果接近；C−P = S−Ke^{−rT}≈2.96 两侧相等
D13
算希腊字母并解释策略
平值 Δ≈0.54、Γ 最大、Θ 为负、V 为正；保护性看跌封底不封顶
D14
四大绩效指标
会算并说清夏普(总风险)/特雷诺(β)/詹森(α)/信息比率(相对基准)适用场景
D15
完整跑通组合流程
配置→构建→再平衡→归因→复盘五步闭环，脚本可复现
完成标准：15 天全部跑通代码且能不看讲义答出上表全部验收项，即达到"金融专业学生水平"的资产定价与组合管理基础；下一步衔接：衍生品定价深入（期权 Greeks 实操）、因子投资实证、或直接进入《公司金融与估值》模块。